# Решения: Практика DP 2D: маршруты и сравнение последовательностей

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    import urllib.request
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    url = (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_09_courier_dp/data/" + name
    )
    dest = Path(name)
    urllib.request.urlretrieve(url, dest)
    return dest.resolve()


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## Урок. 1. Максимальная сумма по сетке

Начните серию с известного шаблона: вправо или вниз, но критерий теперь максимизируется.

In [ ]:
def max_path_sum(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else -10**9
            left = dp[row][col - 1] if col > 0 else -10**9
            dp[row][col] = grid[row][col] + max(top, left)
    return dp[-1][-1]


GRID_A = [[4, 1, 2], [7, 0, 3], [2, 8, 1]]
assert max_path_sum(GRID_A) == 22
assert max_path_sum([[5]]) == 5


## Урок. 2. Восстановить максимальный маршрут

Верните координаты выбранного маршрута. При равенстве разрешён любой из оптимальных вариантов.

In [ ]:
def max_path_route(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[-10**9] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else -10**9
            left = dp[row][col - 1] if col > 0 else -10**9
            dp[row][col] = grid[row][col] + max(top, left)
    row, col = rows - 1, cols - 1
    route = [(row, col)]
    while row or col:
        if row > 0 and (col == 0 or dp[row - 1][col] >= dp[row][col - 1]):
            row -= 1
        else:
            col -= 1
        route.append((row, col))
    return list(reversed(route))


route = max_path_route(GRID_A)
assert route[0] == (0, 0) and route[-1] == (2, 2)
assert sum(GRID_A[row][col] for row, col in route) == 22


## Урок. 3. Число максимальных маршрутов

При равных значениях сверху и слева складывайте количества оптимальных способов.

In [ ]:
def count_max_paths(grid):
    rows, cols = len(grid), len(grid[0])
    best = [[-10**9] * cols for _ in range(rows)]
    count = [[0] * cols for _ in range(rows)]
    best[0][0], count[0][0] = grid[0][0], 1
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            candidates = []
            if row > 0:
                candidates.append((best[row - 1][col], count[row - 1][col]))
            if col > 0:
                candidates.append((best[row][col - 1], count[row][col - 1]))
            previous = max(value for value, _ in candidates)
            best[row][col] = grid[row][col] + previous
            count[row][col] = sum(number for value, number in candidates if value == previous)
    return best[-1][-1], count[-1][-1]


assert count_max_paths([[1, 1], [1, 1]]) == (3, 2)
assert count_max_paths(GRID_A) == (22, 1)


## Урок. 4. Состояние для сравнения строк

`dp[i][j]` — длина общей подпоследовательности префиксов `a[:i]` и `b[:j]`. Создайте таблицу с нулевой рамкой.

In [ ]:
a = "COURIER"
b = "CURSOR"
lcs_dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
assert len(lcs_dp) == len(a) + 1
assert all(len(row) == len(b) + 1 for row in lcs_dp)
assert all(value == 0 for value in lcs_dp[0])


## Урок. 5. Длина общей подпоследовательности

Если последние символы равны, используйте диагональ + 1. Иначе берите максимум сверху и слева.

In [ ]:
def lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


assert lcs_len("COURIER", "CURSOR") == 4
assert lcs_len("", "ABC") == 0
assert lcs_len("ABC", "ABC") == 3


## Урок. 6. Восстановить общую подпоследовательность

Пройдите таблицу назад: равные символы входят в ответ, иначе двигайтесь к соседу с большим значением.

In [ ]:
def lcs_value(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    result = []
    i, j = len(a), len(b)
    while i and j:
        if a[i - 1] == b[j - 1]:
            result.append(a[i - 1])
            i, j = i - 1, j - 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(result))


value = lcs_value("COURIER", "CURSOR")
assert len(value) == 4
assert value == "CURR"


## Урок. 7. Расстояние редактирования

Состояние снова задаётся двумя префиксами. Переход выбирает вставку, удаление или замену одного символа.

In [ ]:
def edit_distance(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            change = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + change,
            )
    return dp[-1][-1]


assert edit_distance("route", "routes") == 1
assert edit_distance("cat", "cut") == 1
assert edit_distance("", "abc") == 3


## Урок. 8. Эксперимент: LCS и расстояние отвечают на разные вопросы

Сравните пары строк. Запишите, почему длинная LCS не всегда означает малое число правок.

In [ ]:
pairs = [("COURIER", "CURSOR"), ("ROUTE", "ROUTES"), ("ABC", "CBA")]
measurements = [(a, b, lcs_len(a, b), edit_distance(a, b)) for a, b in pairs]
COMPARE_NOTE = (
    "LCS измеряет сохранённый порядок символов и допускает пропуски, а расстояние "
    "редактирования считает конкретные операции преобразования. Поэтому две строки "
    "могут иметь заметную общую подпоследовательность, но всё равно требовать "
    "нескольких вставок, удалений или замен."
)
assert len(measurements) == 3
assert all(len(item) == 4 for item in measurements)
assert len(COMPARE_NOTE) >= 120


## Урок. 9. Самостоятельно: наибольшая общая подстрока

В отличие от подпоследовательности, символы должны идти подряд. При несовпадении текущая длина сбрасывается в ноль.

In [ ]:
def longest_common_substring(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    best_length = 0
    best_end = 0
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                if dp[i][j] > best_length:
                    best_length = dp[i][j]
                    best_end = i
    return best_length, a[best_end - best_length:best_end]


length, value = longest_common_substring("COURIER", "CURSOR")
assert (length, value) == (2, "UR")
assert longest_common_substring("ABC", "XYZ") == (0, "")


## ДЗ. A1. Минимальная стоимость новой сетки

Реализуйте 2D-переход на прямоугольной сетке.

In [ ]:
def min_path_cost(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[10**9] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else 10**9
            left = dp[row][col - 1] if col > 0 else 10**9
            dp[row][col] = grid[row][col] + min(top, left)
    return dp[-1][-1]


grid = [[1, 9, 2, 3], [4, 1, 8, 2], [7, 2, 1, 5]]
assert min_path_cost(grid) == 14
assert min_path_cost([[2]]) == 2


## ДЗ. A2. LCS для кодов статусов

Верните длину общей подпоследовательности.

In [ ]:
def lcs_len_hw(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


assert lcs_len_hw("DELIVERED", "DELAYED") == 5
assert lcs_len_hw("", "DELAYED") == 0


## ДЗ. A3. Расстояние между идентификаторами

Посчитайте минимальное число вставок, удалений и замен.

In [ ]:
def edit_distance_hw(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            change = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + change)
    return dp[-1][-1]


assert edit_distance_hw("ORDER", "OLDER") == 1
assert edit_distance_hw("BOX", "") == 3


## ДЗ. B1. Число LCS оптимальной длины

Для строк без повторяющихся символов верните длину LCS и число различных путей таблицы, достигающих этой длины.

In [ ]:
def count_lcs_paths(a, b):
    length = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    count = [[1] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                length[i][j] = length[i - 1][j - 1] + 1
                count[i][j] = count[i - 1][j - 1]
            else:
                length[i][j] = max(length[i - 1][j], length[i][j - 1])
                count[i][j] = 0
                if length[i - 1][j] == length[i][j]:
                    count[i][j] += count[i - 1][j]
                if length[i][j - 1] == length[i][j]:
                    count[i][j] += count[i][j - 1]
    return length[-1][-1], count[-1][-1]


assert count_lcs_paths("ABC", "ACB") == (2, 2)
